In [3]:
import torch 
import torch.nn as nn
import torch.optim as optim
import torchvision 
from torchvision.datasets import CIFAR10

In [10]:
# datasets and dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
transforms=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
trainset=CIFAR10(root='./data', train=True, download=True, transform=transforms)
testset=CIFAR10(root='./data', train=False, download=True, transform=transforms)
trainloader = DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)

testloader = DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

# Build the CNN

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv_layers=nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.fc_layers=nn.Sequential(
            nn.Linear(128*4*4, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)
        return x

        

In [17]:
model = CNN()

In [18]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# Training the CNN

In [23]:
epochs=10
for epoch in range(epochs):
    epoch_training_loss=0.0
    for images , labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images)# forward propogation 
        loss = criterion(output, labels) # loss calculation
        loss.backward() # backward propogation
        optimizer.step() # update weights
        epoch_training_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} & Training Loss: {epoch_training_loss/len(trainloader)}")

Epoch 1/10 & Training Loss: 1.3920658650758015
Epoch 2/10 & Training Loss: 0.966991397959497
Epoch 3/10 & Training Loss: 0.7763926427230201
Epoch 4/10 & Training Loss: 0.6494772873266274
Epoch 5/10 & Training Loss: 0.5390411922350868
Epoch 6/10 & Training Loss: 0.4476441818544322
Epoch 7/10 & Training Loss: 0.3621767081529893
Epoch 8/10 & Training Loss: 0.2909886459998615
Epoch 9/10 & Training Loss: 0.2298071332552168
Epoch 10/10 & Training Loss: 0.18378654839542438


# Evaluate the CNN Model 

In [24]:
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
    for images , labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)
        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f"Test Accuracy: {correct_labels/total_labels*100}%")

Test Accuracy: 75.02%
